# Train Chemprop D-MPNN ensemble on Colab GPU

This notebook clones the `amphiphile` repo, installs deps, and trains a 5-member Chemprop v2 D-MPNN ensemble on the OpenADMET-curated ChEMBL 35 LogD training set. Results are evaluated on the ExpansionRx external benchmark and either uploaded as a GitHub release or downloaded as a zip.

## Before you run

1. **Runtime → Change runtime type → T4 GPU** (free tier is sufficient).
2. Generate a fine-scoped GitHub PAT for the private repo:
   - https://github.com/settings/personal-access-tokens/new
   - Resource owner: `kirkupc`. Repository access: **Only select repositories → amphiphile**.
   - Permissions: **Contents: Read and write**, **Actions: Read**. Nothing else.
   - Expire in 1 day.
   - Copy the token; you'll paste it into the auth cell.

## What you'll end up with

- `models/chemprop/model_{0..4}.pt` — the 5 trained checkpoints.
- `models/chemprop/config.json` — ensemble metadata.
- `models/chemprop_reliability.joblib` — conformal + applicability-domain calibration.
- `reports/chemprop_metrics.json` — scaffold-test + ExpansionRx metrics.
- Optionally: a new GitHub release (`v0.1.0-chemprop`) with these artifacts attached.

## 1. Verify the GPU

In [ ]:
!nvidia-smi | head -20

## 2. Authenticate to the private repo

Paste the PAT when prompted. It's stored in-memory for the session only.

In [ ]:
import os, getpass
os.environ['GITHUB_TOKEN'] = getpass.getpass('GitHub PAT: ')
!echo "$GITHUB_TOKEN" | gh auth login --with-token
!git config --global credential.helper store
!printf 'https://kirkupc:%s@github.com\n' "$GITHUB_TOKEN" > ~/.git-credentials
!git config --global user.email 'colab@amphiphile.local'
!git config --global user.name 'amphiphile-colab'

## 3. Clone the repo

In [ ]:
!git clone https://github.com/kirkupc/amphiphile.git
%cd amphiphile
!git log --oneline | head -5

## 4. Install dependencies

Uses `uv` for fast deterministic installs. Total deps including torch + chemprop + lightning are ~1.5 GB, so this cell takes a few minutes the first time.

In [ ]:
!pip install -q uv
!uv sync --extra dev

## 5. Sanity-check torch + chemprop see the GPU

In [ ]:
!uv run python -c "\
import torch, chemprop, lightning.pytorch as pl; \
print('torch', torch.__version__, 'cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-'); \
print('chemprop', chemprop.__version__); \
print('lightning', pl.__version__)"

## 6. Run training

This is the long step — roughly **15–25 minutes on a T4** for 5 members × 50 epochs × ~18,800 compounds. Go get a coffee.

The `tee` keeps the log on disk so if the cell output gets truncated you can still read it later.

In [ ]:
!uv run logd train --model chemprop 2>&1 | tee chemprop_train.log

## 7. Inspect the metrics

In [ ]:
import json
with open('reports/chemprop_metrics.json') as f:
    m = json.load(f)
print(json.dumps(m, indent=2))

## 8. Publish results — two options

**Option A (recommended):** create a `v0.1.0-chemprop` release with all artifacts attached. Reviewers can download alongside the baseline release. One command, everything lives on GitHub.

**Option B:** commit only `reports/chemprop_metrics.json` back on a new branch, download the larger artifacts as a zip to your laptop, and upload them manually later.

Run the cell for the option you want, not both.

### Option A — GitHub release

In [ ]:
!gh release create v0.1.0-chemprop \
  --title 'v0.1.0 — Chemprop D-MPNN ensemble' \
  --notes 'Chemprop v2 D-MPNN k=5 ensemble trained on the same ChEMBL split as the baseline release. See reports/chemprop_metrics.json for scaffold-test + ExpansionRx numbers.' \
  models/chemprop/model_0.pt \
  models/chemprop/model_1.pt \
  models/chemprop/model_2.pt \
  models/chemprop/model_3.pt \
  models/chemprop/model_4.pt \
  models/chemprop/config.json \
  models/chemprop_reliability.joblib \
  reports/chemprop_metrics.json

### Option B — commit metrics on a branch, download artifacts

In [ ]:
!git checkout -b chemprop-results
!git add reports/chemprop_metrics.json
!git commit -m 'Add Chemprop D-MPNN metrics from Colab run'
!git push -u origin chemprop-results

from google.colab import files
import shutil
shutil.make_archive('chemprop_artifacts', 'zip', 'models/chemprop')
files.download('chemprop_artifacts.zip')
files.download('models/chemprop_reliability.joblib')

## Troubleshooting

**Chemprop or torch install fails in step 4.** Fresh Colab images sometimes ship a different CUDA driver than the torch wheel wants. Fallback:

```python
!pip install 'torch>=2.2' 'chemprop>=2.0' 'lightning>=2.0'
!uv sync --extra dev --no-install-project
!uv pip install -e .
```

**Training crashes with OOM.** Reduce `batch_size` in `train_chemprop()` or drop `k` to 3 for a smoke test:

```python
!uv run logd train --model chemprop --k 3
```

**Training silently hangs after epoch 1.** Almost always a Lightning namespace clash (`lightning.pytorch` vs `pytorch_lightning`). Our `chemprop_wrap.py` uses the correct one, but if Colab's default environment imports the other elsewhere, kill the runtime (Runtime → Disconnect and delete runtime) and restart with a clean image.

**PAT auth fails.** The token needs `Contents: Read and write` — `Read` alone will clone but not push. Re-generate with both perms.

**`gh release create` says release exists.** You've already run this notebook today. Either `gh release delete v0.1.0-chemprop` first, or use a new tag like `v0.1.0-chemprop-2`.